In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('data_cleaning/[복지]_문화시설_총합.csv')

In [7]:
df = df.dropna(subset=['이름'])

In [34]:
df[df['법정동'].isna()]

,Unnamed: 0,시도,시군구,이름,주소,법정동,행정동


In [ ]:
df.loc[777,  ['법정동', '행정동']] = ('대치동','신흥동')

In [32]:
df = df.dropna(subset=['법정동'])

In [39]:
df = df.iloc[:,1:]

In [41]:
df.to_csv('data_cleaning/[복지]_문화시설_총합_최종.csv')

In [43]:
df_center = pd.read_csv('data_cleaning/[복지]통합지역아동센터.csv')

In [45]:
df_center = df_center.iloc[:,1:]

In [46]:
df_center.head()

,시설명,주소,Y좌표값,X좌표값,행정동
0,새꿈터지역아동센터,경기도 고양시 일산동구 산두로 229번길 29,37.674761,126.778352,정발산동
1,맑은샘지역아동센터,경기도 고양시 일산동구 고봉로 658번길 13,37.715616,126.789983,성석동
2,한빛지역아동센터,경기도 고양시 덕양구 행당로 11번길 11 5층,37.620404,126.825960,토당동
3,푸른학교반디교실지역아동센터,경기도 고양시 덕양구 토당로 78번길 7 22 2층,37.623210,126.819300,토당동
4,신성지역아동센터,경기도 고양시 일산동구 일산로 11,37.641440,126.784387,백석동


In [47]:
import requests
import pandas as pd
import time

url = "https://dapi.kakao.com/v2/local/search/address.json"
headers = {"Authorization": "KakaoAK YOUR_KAKAO_API_KEY"}

def get_address(address):
    try:
        params = {'query' : address}
        res = requests.get(url, headers=headers, params=params, timeout=5)
        a = res.json()
        law_dong = a['documents'][0]['address']['region_3depth_name']   # 법정동
        hang_dong = a['documents'][0]['address']['region_3depth_h_name']   # 행정동
        return pd.Series([law_dong, hang_dong])
        
    except Exception as e:
        print(f"주소 변환 실패: {address} / 에러: {e}")
        return pd.Series([None, None])



In [ ]:
import requests
import pandas as pd
import time

url = "https://dapi.kakao.com/v2/local/geo/coord2address.json"
headers = {"Authorization": "KakaoAK YOUR_KAKAO_API_KEY"}

def get_dong(xy):
    x = xy['WGS84경도']
    y = xy['WGS84위도']
    try:
        params = {'x' : x, 'y' : y}
        res = requests.get(url, headers=headers, params=params, timeout=5)
        a = res.json()

        lat = a['documents'][0]['address']['region_3depth_name']
        return lat

    except Exception as e:
        print(f"주소 변환 실패: {xy} / 에러: {e}")
        return None


df['행정동'] = df[['WGS84경도', 'WGS84위도']].apply(get_dong, axis=1)

In [48]:
df_center[['법정동', '행정동']] = df_center['주소'].apply(get_address)


주소 변환 실패: 경기도 구리시 아치울길23번길 23 9 / 에러: list index out of range
주소 변환 실패: 경기도 김포시 돌문로95번길 12 1 사우동 2층 / 에러: list index out of range
주소 변환 실패: 경기도 남양주시 다산순환로 126 301 302호 / 에러: list index out of range
주소 변환 실패: 경기도 남양주시 진건읍 진건오남로105번길 6 301 302호 / 에러: list index out of range
주소 변환 실패: 경기도 부천시 오정구 상오정로 611 / 에러: list index out of range
주소 변환 실패: 경기도 용인시 수지구 광교마을로 90 1 상현동 휴먼시아아파트 41단지 주민복지관1 / 에러: list index out of range
주소 변환 실패: 경기도 용인시 기흥구 용구대로2152번길 34 202 203호 / 에러: list index out of range
주소 변환 실패: 서울특별시 동작구 장승배기로10가길 1 / 에러: list index out of range
주소 변환 실패: 서울특별시 강동구 올림픽로98길 25 / 에러: list index out of range
주소 변환 실패: 서울특별시 성북구 동소문로42나길 26-15 / 에러: list index out of range
주소 변환 실패: 인천광역시 봉오대로698번길 11, 2~3층(작전동) / 에러: list index out of range
주소 변환 실패: 인천광역시 안남로573번길 26, 401~404호(효성동) / 에러: list index out of range
주소 변환 실패: 인천광역시 장제로 997번길2, 2~3층(박촌동) / 에러: list index out of range


In [69]:
df_center[df_center['법정동'].isna()]

,시설명,주소,Y좌표값,X좌표값,행정동,법정동
675,월곡 지역아동센터,서울특별시 성북구 동소문로42나길 26-15,37.605130,127.029630,None,None
838,인천행복한돌봄계양 지역아동센터,"인천광역시 봉오대로698번길 11, 2~3층(작전동)",37.528816,126.726442,None,None
852,청운지역아동센터,"인천광역시 안남로573번길 26, 401~404호(효성동)",37.534673,126.708625,None,None
861,소양지역아동센터,"인천광역시 장제로 997번길2, 2~3층(박촌동)",37.554707,126.745150,None,None


In [79]:
df_center

,시설명,주소,Y좌표값,X좌표값,행정동,법정동
0,새꿈터지역아동센터,경기도 고양시 일산동구 산두로 229번길 29,37.674761,126.778352,정발산동,정발산동
1,맑은샘지역아동센터,경기도 고양시 일산동구 고봉로 658번길 13,37.715616,126.789983,고봉동,성석동
2,한빛지역아동센터,경기도 고양시 덕양구 행당로 11번길 11 5층,37.620404,126.825960,행주동,토당동
3,푸른학교반디교실지역아동센터,경기도 고양시 덕양구 토당로 78번길 7 22 2층,37.623210,126.819300,능곡동,토당동
4,신성지역아동센터,경기도 고양시 일산동구 일산로 11,37.641440,126.784387,백석2동,백석동
...,...,...,...,...,...,...
896,화도마리지역아동센터,인천광역시 화도면 가능포로 37-12(상방리),37.635953,126.417742,화도면,화도면 상방리
897,희망터지역아동센터,인천광역시 길상면 온수길38번길 14(온수리),37.640518,126.489703,길상면,길상면 온수리
898,강화지역아동센터,"인천광역시 강화읍 갑룡길117번길 28-3, 나동 102호(용정리)",37.747141,126.505036,강화읍,강화읍 용정리
899,영흥지역아동센터,인천광역시 영흥면 영흥서로 220,37.274026,126.478676,영흥면,영흥면 내리


In [78]:
df_center = df_center[df_center['주소'] != '서울특별시 성북구 동소문로42나길 26-15']

In [80]:
df_center.to_csv('data_cleaning/[복지]통합지역아동센터_최종.csv')